# Lesson 3: Chatbot Example

In this lesson, you will familiarize yourself with the chatbot example you will work on during this course. The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.

## Import Libraries

In [1]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic
from mcp.server.fastmcp import FastMCP

ModuleNotFoundError: No module named 'mcp'

## Tool Functions

In [15]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  

In [16]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    
    # Use arxiv to find the papers 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info  
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids

In [17]:
search_papers("computers")

Results are saved in: papers/computers/papers_info.json


['1310.7911v2',
 'math/9711204v1',
 '2208.00733v1',
 '2504.07020v1',
 '2403.03925v1']

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.

In [18]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
 
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."

In [19]:
extract_info('1310.7911v2')

'{\n  "title": "Compact manifolds with computable boundaries",\n  "authors": [\n    "Zvonko Iljazovic"\n  ],\n  "summary": "We investigate conditions under which a co-computably enumerable closed set\\nin a computable metric space is computable and prove that in each locally\\ncomputable computable metric space each co-computably enumerable compact\\nmanifold with computable boundary is computable. In fact, we examine the notion\\nof a semi-computable compact set and we prove a more general result: in any\\ncomputable metric space each semi-computable compact manifold with computable\\nboundary is computable. In particular, each semi-computable compact\\n(boundaryless) manifold is computable.",\n  "pdf_url": "http://arxiv.org/pdf/1310.7911v2",\n  "published": "2013-10-29"\n}'

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [20]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping

This code handles tool mapping and execution.

In [21]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
    
    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [22]:
load_dotenv() 
client = anthropic.Anthropic()

### Query Processing

In [23]:
def process_query(query):
    
    messages = [{'role': 'user', 'content': query}]
    
    response = client.messages.create(max_tokens = 2024,
                                  model = 'claude-3-7-sonnet-20250219', 
                                  tools = tools,
                                  messages = messages)
    
    process_query = True
    while process_query:
        assistant_content = []

        for content in response.content:
            if content.type == 'text':
                
                print(content.text)
                assistant_content.append(content)
                
                if len(response.content) == 1:
                    process_query = False
            
            elif content.type == 'tool_use':
                
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({"role": "user", 
                                  "content": [
                                      {
                                          "type": "tool_result",
                                          "tool_use_id": tool_id,
                                          "content": result
                                      }
                                  ]
                                })
                response = client.messages.create(max_tokens = 2024,
                                  model = 'claude-3-7-sonnet-20250219', 
                                  tools = tools,
                                  messages = messages) 
                
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query = False

### Chat Loop

In [24]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"

To access the `papers` folder: 1) click on the `File` option on the top menu of the notebook and 2) click on `Open` and then 3) click on `L3`.

In [25]:
chat_loop()

Type your queries or 'quit' to exit.

Error: create() got an unexpected keyword argument 'tools'


<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b> To Access the <code>requirements.txt</code> file or the <code>papers</code> folder: </b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em> and finally 3) click on <em>"L3"</em>.
</div>

In the next lessons, you will take out the tool definitions to wrap them in an MCP server. Then you will create an MCP client inside the chatbot to make the chatbot MCP compatible.  

## Resources

[Guide on how to implement tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview#how-to-implement-tool-use)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">


<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

</div>

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Python path: {sys.path[:3]}...")  # First 3 paths

In [ ]:
!pip install arxiv

In [ ]:
import arxiv
print("arxiv module successfully imported!")
print(f"arxiv version: {arxiv.__version__}")

In [ ]:
!pip install python-dotenv

In [ ]:
!pip install anthropic

In [ ]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

In [ ]:
!pip install --upgrade typing-extensions

In [ ]:
!pip install "anthropic<0.26" --force-reinstall

In [26]:
import anthropic
print(f"Anthropic version: {anthropic.__version__}")

Anthropic version: 0.25.9


In [27]:
!pip install --upgrade anthropic

  Using cached anthropic-0.38.0-py3-none-any.whl (951 kB)
  Using cached anthropic-0.37.1-py3-none-any.whl (945 kB)
  Using cached anthropic-0.37.0-py3-none-any.whl (945 kB)
  Using cached anthropic-0.36.2-py3-none-any.whl (939 kB)
  Using cached anthropic-0.36.1-py3-none-any.whl (939 kB)
  Using cached anthropic-0.36.0-py3-none-any.whl (939 kB)
  Using cached anthropic-0.35.0-py3-none-any.whl (894 kB)
  Using cached anthropic-0.34.2-py3-none-any.whl (891 kB)
  Using cached anthropic-0.34.1-py3-none-any.whl (891 kB)
  Using cached anthropic-0.34.0-py3-none-any.whl (891 kB)
  Using cached anthropic-0.33.1-py3-none-any.whl (866 kB)
  Using cached anthropic-0.33.0-py3-none-any.whl (866 kB)
  Using cached anthropic-0.32.0-py3-none-any.whl (866 kB)
  Using cached anthropic-0.31.2-py3-none-any.whl (865 kB)
  Using cached anthropic-0.31.1-py3-none-any.whl (865 kB)
  Using cached anthropic-0.31.0-py3-none-any.whl (865 kB)
  Using cached anthropic-0.30.1-py3-none-any.whl (863 kB)
  Using cached

In [28]:
import anthropic
print(f"New Anthropic version: {anthropic.__version__}")

New Anthropic version: 0.25.9


In [29]:
import importlib
import anthropic
importlib.reload(anthropic)
print(f"Reloaded Anthropic version: {anthropic.__version__}")

Reloaded Anthropic version: 0.25.9


In [30]:
# Reimport everything with the updated library
import sys
sys.modules.pop('anthropic', None)  # Remove cached module

import anthropic
print(f"Anthropic version: {anthropic.__version__}")

# Test if the tools parameter is supported
try:
    client = anthropic.Anthropic()
    print("Client created successfully!")
except Exception as e:
    print(f"Error creating client: {e}")

Anthropic version: 0.25.9
Client created successfully!


In [31]:
%reset -f

In [32]:
import anthropic
print(f"Anthropic version: {anthropic.__version__}")

Anthropic version: 0.25.9


In [33]:
# Let's recreate the working functions with the correct API for version 0.25.9

import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

# Tool functions (same as before)
PAPER_DIR = "papers"

def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """Search for papers on arXiv based on a topic and store their information."""
    client = arxiv.Client()
    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    papers = client.results(search)
    
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    return paper_ids

def extract_info(paper_id: str) -> str:
    """Search for information about a specific paper across all topic directories."""
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."

# Tool schema and mapping
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    result = mapping_tool_function[tool_name](**tool_args)
    if result is None:
        result = "The operation completed but didn't return any results."
    elif isinstance(result, list):
        result = ', '.join(result)
    elif isinstance(result, dict):
        result = json.dumps(result, indent=2)
    else:
        result = str(result)
    return result

print("Functions loaded successfully!")

Functions loaded successfully!


In [34]:
# Fixed process_query function for anthropic 0.25.9
load_dotenv() 
client = anthropic.Anthropic()

def process_query(query):
    messages = [{'role': 'user', 'content': query}]
    
    # For anthropic 0.25.9, use the correct parameter name
    response = client.messages.create(
        max_tokens=2024,
        model='claude-3-haiku-20240307',  # Using a model that should work
        tools=tools,  # This should work in 0.26+, let's test if it's actually updated
        messages=messages
    )
    
    process_query_flag = True
    while process_query_flag:
        assistant_content = []

        for content in response.content:
            if content.type == 'text':
                print(content.text)
                assistant_content.append(content)
                
                if len(response.content) == 1:
                    process_query_flag = False
            
            elif content.type == 'tool_use':
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({
                    "role": "user", 
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_id,
                            "content": result
                        }
                    ]
                })
                
                response = client.messages.create(
                    max_tokens=2024,
                    model='claude-3-haiku-20240307', 
                    tools=tools,
                    messages=messages
                ) 
                
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query_flag = False

print("process_query function updated!")

process_query function updated!


In [39]:
# Test the fixed function with a simple query
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

# Test with a simple query first
try:
    process_query("Hi")
except Exception as e:
    print(f"Error details: {e}")
    print("The issue might still be with the anthropic library version.")

Error details: create() got an unexpected keyword argument 'tools'
The issue might still be with the anthropic library version.


In [44]:
chat_loop()

Type your queries or 'quit' to exit.

Error: create() got an unexpected keyword argument 'tools'


In [41]:
# Fixed version for anthropic 0.25.9 without tools parameter
def process_query_old_version(query):
    """Version compatible with anthropic 0.25.9"""
    messages = [{'role': 'user', 'content': query}]
    
    # For older versions, we need to handle tool calls differently
    # Let's first try without tools and see what the model says
    response = client.messages.create(
        max_tokens=2024,
        model='claude-3-haiku-20240307',
        messages=messages
    )
    
    # Print the response
    for content in response.content:
        if content.type == 'text':
            print(content.text)

# Test the simplified version
try:
    process_query_old_version("Hi")
    print("✓ Basic chat works!")
except Exception as e:
    print(f"Error: {e}")
    print("Let's check what models are available...")

Error: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Let's check what models are available...


In [42]:
# Check if .env file exists and what's in it
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is loaded
api_key = os.getenv('ANTHROPIC_API_KEY')
if api_key:
    print("✓ ANTHROPIC_API_KEY found in environment")
    print(f"Key starts with: {api_key[:10]}...")
else:
    print("✗ ANTHROPIC_API_KEY not found in environment")
    print("You need to:")
    print("1. Create a .env file in your directory")
    print("2. Add: ANTHROPIC_API_KEY=your_actual_api_key")
    print("3. Or set the environment variable directly")

✓ ANTHROPIC_API_KEY found in environment
Key starts with: sk-ant-api...


In [43]:
# Option 1: Set API key directly (replace with your actual key)
# os.environ['ANTHROPIC_API_KEY'] = 'your_actual_api_key_here'

# Option 2: Check if .env file exists
if os.path.exists('.env'):
    print("✓ .env file exists")
    with open('.env', 'r') as f:
        content = f.read()
        if 'ANTHROPIC_API_KEY' in content:
            print("✓ ANTHROPIC_API_KEY found in .env file")
        else:
            print("✗ ANTHROPIC_API_KEY not found in .env file")
else:
    print("✗ No .env file found")
    print("Create a .env file with: ANTHROPIC_API_KEY=your_key_here")

✓ .env file exists
✓ ANTHROPIC_API_KEY found in .env file


In [45]:
# Reload the environment variables
import importlib
import os
from dotenv import load_dotenv

# Force reload of the .env file
load_dotenv(override=True)

# Check if API key is now loaded
api_key = os.getenv('ANTHROPIC_API_KEY')
if api_key:
    print("✓ ANTHROPIC_API_KEY loaded successfully")
    print(f"Key starts with: {api_key[:15]}...")
    
    # Now test the anthropic client
    import anthropic
    
    try:
        client = anthropic.Anthropic(api_key=api_key)
        print("✓ Anthropic client created successfully")
        
        # Test a simple message (without tools for now)
        response = client.messages.create(
            max_tokens=50,
            model='claude-3-haiku-20240307',
            messages=[{'role': 'user', 'content': 'Hi, just say hello back'}]
        )
        
        print("✓ API connection test successful!")
        for content in response.content:
            if content.type == 'text':
                print(f"Response: {content.text}")
                
    except Exception as e:
        print(f"✗ Error testing API: {e}")
else:
    print("✗ API key still not found")

✓ ANTHROPIC_API_KEY loaded successfully
Key starts with: sk-ant-api03-jh...
✓ Anthropic client created successfully
✓ API connection test successful!
Response: Hello!


In [46]:
# Test if tools parameter works with current version
try:
    # Test with tools parameter
    response = client.messages.create(
        max_tokens=100,
        model='claude-3-haiku-20240307',
        tools=tools,  # This is what was causing the error
        messages=[{'role': 'user', 'content': 'Hi'}]
    )
    print("✓ Tools parameter works! Your anthropic version supports it.")
    for content in response.content:
        if content.type == 'text':
            print(f"Response: {content.text}")
            
except Exception as e:
    print(f"✗ Tools parameter error: {e}")
    print("Your anthropic version doesn't support 'tools' parameter")
    print("Let's check your actual version...")
    
    import anthropic
    print(f"Current version: {anthropic.__version__}")
    
    if anthropic.__version__ < "0.26.0":
        print("You need to upgrade to anthropic >= 0.26.0")
        print("Run: !pip install --upgrade anthropic")
    else:
        print("Version should support tools, there might be another issue")

✗ Tools parameter error: create() got an unexpected keyword argument 'tools'
Your anthropic version doesn't support 'tools' parameter
Let's check your actual version...
Current version: 0.25.9
You need to upgrade to anthropic >= 0.26.0
Run: !pip install --upgrade anthropic


In [47]:
!pip install --upgrade anthropic

  Using cached anthropic-0.38.0-py3-none-any.whl (951 kB)
  Using cached anthropic-0.37.1-py3-none-any.whl (945 kB)
  Using cached anthropic-0.37.0-py3-none-any.whl (945 kB)
  Using cached anthropic-0.36.2-py3-none-any.whl (939 kB)
  Using cached anthropic-0.36.1-py3-none-any.whl (939 kB)
  Using cached anthropic-0.36.0-py3-none-any.whl (939 kB)
  Using cached anthropic-0.35.0-py3-none-any.whl (894 kB)
  Using cached anthropic-0.34.2-py3-none-any.whl (891 kB)
  Using cached anthropic-0.34.1-py3-none-any.whl (891 kB)
  Using cached anthropic-0.34.0-py3-none-any.whl (891 kB)
  Using cached anthropic-0.33.1-py3-none-any.whl (866 kB)
  Using cached anthropic-0.33.0-py3-none-any.whl (866 kB)
  Using cached anthropic-0.32.0-py3-none-any.whl (866 kB)
  Using cached anthropic-0.31.2-py3-none-any.whl (865 kB)
  Using cached anthropic-0.31.1-py3-none-any.whl (865 kB)
  Using cached anthropic-0.31.0-py3-none-any.whl (865 kB)
  Using cached anthropic-0.30.1-py3-none-any.whl (863 kB)
  Using cached

In [48]:
# Restart kernel and reimport with updated version
%reset -f

# Reimport everything
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

print(f"New Anthropic version: {anthropic.__version__}")

# Reload environment
load_dotenv(override=True)

# Create client
client = anthropic.Anthropic()

# Tool functions and schema (same as before)
PAPER_DIR = "papers"

def search_papers(topic: str, max_results: int = 5) -> List[str]:
    client = arxiv.Client()
    search = arxiv.Search(query=topic, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    papers = client.results(search)
    
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    file_path = os.path.join(path, "papers_info.json")
    
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    return paper_ids

def extract_info(paper_id: str) -> str:
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError):
                    continue
    return f"There's no saved information related to paper {paper_id}."

tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "The topic to search for"}, 
                "max_results": {"type": "integer", "description": "Maximum number of results to retrieve", "default": 5}
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {"paper_id": {"type": "string", "description": "The ID of the paper to look for"}},
            "required": ["paper_id"]
        }
    }
]

mapping_tool_function = {"search_papers": search_papers, "extract_info": extract_info}

def execute_tool(tool_name, tool_args):
    result = mapping_tool_function[tool_name](**tool_args)
    if result is None:
        result = "The operation completed but didn't return any results."
    elif isinstance(result, list):
        result = ', '.join(result)
    elif isinstance(result, dict):
        result = json.dumps(result, indent=2)
    else:
        result = str(result)
    return result

print("✓ All functions loaded successfully!")

New Anthropic version: 0.25.9
✓ All functions loaded successfully!


In [49]:
import sys
print("Before removing module:", sys.modules.get('anthropic', 'Not found'))

# Remove from cache
if 'anthropic' in sys.modules:
    del sys.modules['anthropic']
    
# Clear all anthropic-related modules
modules_to_remove = [key for key in sys.modules.keys() if 'anthropic' in key.lower()]
for module in modules_to_remove:
    del sys.modules[module]
    
print("Removed modules:", modules_to_remove)

# Force reimport
import anthropic
print(f"After reimport: {anthropic.__version__}")

# Try to create client and test tools
client = anthropic.Anthropic()

# Test tools parameter
try:
    response = client.messages.create(
        max_tokens=50,
        model='claude-3-haiku-20240307',
        tools=tools,
        messages=[{'role': 'user', 'content': 'Hi'}]
    )
    print("✓ Tools parameter now works!")
    for content in response.content:
        if content.type == 'text':
            print(f"Response: {content.text}")
except Exception as e:
    print(f"✗ Still getting error: {e}")

Before removing module: <module 'anthropic' from '/Users/wenlin/.pyenv/versions/3.7.13/lib/python3.7/site-packages/anthropic/__init__.py'>
Removed modules: ['anthropic.types', 'anthropic.types.usage', 'anthropic._models', 'anthropic._types', 'anthropic._utils', 'anthropic._utils._sync', 'anthropic._utils._proxy', 'anthropic._utils._utils', 'anthropic._compat', 'anthropic._utils._typing', 'anthropic._utils._streams', 'anthropic._utils._transform', 'anthropic._files', 'anthropic._constants', 'anthropic.types.message', 'anthropic.types.text_block', 'anthropic.types.content_block', 'anthropic.types.completion', 'anthropic.types.text_delta', 'anthropic.types.message_param', 'anthropic.types.text_block_param', 'anthropic.types.image_block_param', 'anthropic.types.message_stop_event', 'anthropic.types.message_delta_event', 'anthropic.types.message_delta_usage', 'anthropic.types.message_start_event', 'anthropic.types.message_stream_event', 'anthropic.types.content_block_stop_event', 'anthropic

In [50]:
# Let's check what's actually installed and use the working method for the current setup
import subprocess
result = subprocess.run(['pip', 'show', 'anthropic'], capture_output=True, text=True)
print("Pip show anthropic:")
print(result.stdout)

# Let's use the beta tools interface which might be available
try:
    # Try the beta tools approach
    from anthropic import Anthropic
    client = Anthropic()
    
    # Use beta tools if available
    message = client.beta.tools.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=50,
        tools=tools,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print("✓ Beta tools works!")
    for content in message.content:
        if hasattr(content, 'text'):
            print(f"Response: {content.text}")
            
except Exception as e:
    print(f"Beta tools error: {e}")
    
    # Let's try without tools parameter and create a working chat
    print("Creating basic chat without tools...")
    try:
        response = client.messages.create(
            max_tokens=100,
            model='claude-3-haiku-20240307',
            messages=[{'role': 'user', 'content': 'Hi'}]
        )
        print("✓ Basic chat works!")
        for content in response.content:
            if content.type == 'text':
                print(f"Response: {content.text}")
    except Exception as e2:
        print(f"Basic chat error: {e2}")

Pip show anthropic:
Name: anthropic
Version: 0.26.0
Summary: The official Python library for the anthropic API
Home-page: 
Author: 
Author-email: Anthropic <support@anthropic.com>
License: 
Location: /Users/wenlin/.pyenv/versions/3.7.13/lib/python3.7/site-packages
Requires: anyio, cached-property, distro, httpx, pydantic, sniffio, tokenizers, typing-extensions
Required-by: 

✓ Beta tools works!
Response: Hello! How can I assist you today?


In [55]:
# Working process_query function using beta tools
def process_query(query):
    messages = [{'role': 'user', 'content': query}]
    
    # Use beta tools API
    response = client.beta.tools.messages.create(
        max_tokens=2024,
        model='claude-3-haiku-20240307', 
        tools=tools,
        messages=messages
    )
    
    process_query_flag = True
    while process_query_flag:
        assistant_content = []

        for content in response.content:
            if content.type == 'text':
                print(content.text)
                assistant_content.append(content)
                
                if len(response.content) == 1:
                    process_query_flag = False
            
            elif content.type == 'tool_use':
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({
                    "role": "user", 
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_id,
                            "content": result
                        }
                    ]
                })
                
                response = client.beta.tools.messages.create(
                    max_tokens=2024,
                    model='claude-3-haiku-20240307', 
                    tools=tools,
                    messages=messages
                ) 
                
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query_flag = False

def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

print("✓ Fixed functions ready!")
print("Now you can run: chat_loop()")

✓ Fixed functions ready!
Now you can run: chat_loop()


In [56]:
chat_loop()

Type your queries or 'quit' to exit.
Okay, let's search for papers related to Rubik's Cube on arXiv:
Calling tool search_papers with args {'topic': 'rubik cube'}
Results are saved in: papers/rubik_cube/papers_info.json
Found 5 papers with IDs: 2203.02780v1, 2502.13518v1, 1706.06708v2, 2112.08602v1, 2301.12167v1
The search returned 5 relevant papers on arXiv related to Rubik's Cube. Let me provide some high-level information about these papers:
Calling tool extract_info with args {'paper_id': '2203.02780v1'}
Calling tool extract_info with args {'paper_id': '2502.13518v1'}
The search results cover a range of topics related to Rubik's Cube, including analysis of higher-dimensional analogues, theoretical properties of the cube's group of configurations, and mathematical generalizations. Let me know if you need any clarification or have additional questions!




In [53]:
# Updated search_papers function to display paper IDs
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """Search for papers on arXiv based on a topic and store their information."""
    client = arxiv.Client()
    search = arxiv.Search(query=topic, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    papers = client.results(search)
    
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    file_path = os.path.join(path, "papers_info.json")
    
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_id = paper.get_short_id()
        paper_ids.append(paper_id)
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper_id] = paper_info
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    print(f"Found {len(paper_ids)} papers with IDs: {', '.join(paper_ids)}")
    
    return paper_ids

# Update the tool mapping
mapping_tool_function = {"search_papers": search_papers, "extract_info": extract_info}

print("✓ Updated search_papers function - now displays paper IDs!")

✓ Updated search_papers function - now displays paper IDs!


In [54]:
# Test the updated function
try:
    # Test the search_papers function directly
    result = search_papers("quantum computing", 3)
    print(f"Function returned: {result}")
    
except Exception as e:
    print(f"Error: {e}")
    
print("\n" + "="*50)
print("Now test through the chatbot:")
print("="*50)

# Test through the chatbot
process_query("Search for 3 papers on deep learning")

Results are saved in: papers/quantum_computing/papers_info.json
Found 3 papers with IDs: 2208.00733v1, quant-ph/0003151v1, 1311.4939v1
Function returned: ['2208.00733v1', 'quant-ph/0003151v1', '1311.4939v1']

Now test through the chatbot:
Okay, let's search for 3 papers on the topic of deep learning:
Calling tool search_papers with args {'topic': 'deep learning', 'max_results': 3}
Results are saved in: papers/deep_learning/papers_info.json
Found 3 papers with IDs: 1805.08355v1, 1806.01756v1, 1908.02130v1
The search returned the arXiv IDs for 3 papers on deep learning. I can now try to extract more information about each of these papers.


In [62]:
# Current working imports - no MCP needed yet
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic
from mcp.server.fastmcp import FastMCP

print("✓ All required modules loaded successfully!")
print("✓ No MCP module needed for this lesson")
print("\nCurrent working chatbot uses:")
print("- arxiv: for paper search")
print("- anthropic: for Claude API")
print("- json: for data storage") 
print("- dotenv: for environment variables")
print("\nMCP will be introduced in later lessons!")

ModuleNotFoundError: No module named 'mcp'

In [2]:
!pip install mcp

ERROR: Could not find a version that satisfies the requirement mcp (from versions: none)
ERROR: No matching distribution found for mcp
You should consider upgrading via the '/Users/wenlin/.pyenv/versions/3.7.13/bin/python3.7 -m pip install --upgrade pip' command.


In [3]:
!pip install model-context-protocol

ERROR: Could not find a version that satisfies the requirement model-context-protocol (from versions: none)
ERROR: No matching distribution found for model-context-protocol
You should consider upgrading via the '/Users/wenlin/.pyenv/versions/3.7.13/bin/python3.7 -m pip install --upgrade pip' command.


In [4]:
!pip search mcp

ERROR: XMLRPC request failed [code: -32500]
RuntimeError: PyPI no longer supports 'pip search' (or XML-RPC search). Please use https://pypi.org/search (via a browser) instead. See https://warehouse.pypa.io/api-reference/xml-rpc.html#deprecated-methods for more information.


In [5]:
!pip install mcp-server-tools

ERROR: Could not find a version that satisfies the requirement mcp-server-tools (from versions: none)
ERROR: No matching distribution found for mcp-server-tools
You should consider upgrading via the '/Users/wenlin/.pyenv/versions/3.7.13/bin/python3.7 -m pip install --upgrade pip' command.


In [6]:
!pip install anthropic-mcp

ERROR: Could not find a version that satisfies the requirement anthropic-mcp (from versions: none)
ERROR: No matching distribution found for anthropic-mcp
You should consider upgrading via the '/Users/wenlin/.pyenv/versions/3.7.13/bin/python3.7 -m pip install --upgrade pip' command.
